# Managed MCP: Firestore Server Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-feb-2026/mcp_firestore_demo.ipynb)

This notebook demonstrates how to use the **Firestore Remote MCP (Model Context Protocol) Server** to allow an AI agent to interact with Firestore documents as 'tools'.

## Use Case
A customer support agent needs to retrieve and update user preferences stored in Firestore. Instead of writing custom API code, the agent uses the standard MCP protocol to 'discover' and 'use' Firestore as a tool.

### Requirements
- BigQuery/Firestore APIs enabled.
- **Managed MCP**: The `firestore-mcp.googleapis.com` service must be enabled.

In [ ]:
!pip install google-cloud-firestore google-adk
from google.colab import auth
auth.authenticate_user()

import os
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
firestore_instance = '(default)' # @param {type:"string"}

### 2. [MANDATORY] Enable Managed MCP Service

To use the managed Firestore MCP server, you must explicitly enable the specialized service endpoint.

In [ ]:
# Enable the Firestore MCP service
!gcloud services enable firestore-mcp.googleapis.com --project={project_id}
print("Success: Managed Firestore MCP service enabled.")

### 3. [PREREQUISITES] Create Firestore Sample Data

This block ensures a Firestore database exists and contains sample documents for the agent to query.

In [ ]:
from google.cloud import firestore

def setup_firestore():
    db = firestore.Client(project=project_id, database=firestore_instance)
    
    # Create Sample Data in 'subscriptions' collection
    db.collection('subscriptions').document('user_456').set({
        'user_id': 'user_456',
        'status': 'Active',
        'tier': 'Premium',
        'renewal_date': '2026-04-12'
    })
    print("Sample data successfully created in Firestore.")

setup_firestore()

### 4. Configure the ADK Agent with MCP Tool

The agent is configured to use the **Gemini 3.1 Pro** model and the Managed MCP endpoint.

In [ ]:
import google.auth
from google.auth.transport.requests import Request
from google.adk import Agent, Runner
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams
from google.adk.sessions.in_memory_session_service import InMemorySessionService

# Set environment for Vertex AI
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

# Refresh credentials for the MCP connection
scopes = ["https://www.googleapis.com/auth/cloud-platform"]
creds, _ = google.auth.default(scopes=scopes)
creds.refresh(Request())

# Initialize Firestore MCP Toolset
firestore_mcp_toolset = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url="https://firestore-mcp.googleapis.com/v1",
        headers={"Authorization": f"Bearer {creds.token}"},
        timeout=30.0 
    )
)

agent = Agent(
    model="gemini-3.1-pro-preview",
    name="SupportAssistant",
    instruction="You are a customer support agent. Use Firestore tools to look up user data when asked.",
    tools=[firestore_mcp_toolset]
)

runner = Runner(agent=agent, session_service=InMemorySessionService(), app_name="FirestoreDemo", auto_create_session=True)
print("Agent Initialized with high-fidelity McpToolset and Gemini 3.1 Pro Preview")

### 5. Test the Agent
The agent will dynamically discover and execute Firestore tools via the MCP protocol.

In [ ]:
import asyncio
from google.genai import types

user_query = "What is the current subscription status for user 'user_456'?"

async def run_support_demo():
    message = types.Content(parts=[types.Part(text=user_query)], role='user')
    response_stream = runner.run_async(user_id="lead_support", session_id="s1", new_message=message)
    
    print("--- Agent Execution ---")
    async for event in response_stream:
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text: print(f"Agent: {part.text}")
                if part.function_call: print(f"[SYSTEM]: Calling Firestore tool '{part.function_call.name}'")

await run_support_demo()

### 6. Things to remember or know
- **Standardized Interface**: MCP provides a common protocol for Firestore, BigQuery, and custom APIs, allowing agents to use the same `McpToolset` regardless of the backend.
- **Real-time Interaction**: Gemini 3.1 Pro's 2026 orchestration layer is designed for low-latency, recursive tool calling to explore document hierarchies on-demand.
- **Enterprise Security**: Managed MCP servers enforce document-level access controls through standard IAM, ensuring data perimeter security within the agent session.